# Minimal Diffusion Transformer in JAX/Flax with `jaxpr` and StableHLO

This notebook implements a compact **Diffusion Transformer (DiT-style)** model in **JAX + Flax**, then shows how JAX transforms the computation through:

1. Python / Flax module definition
2. JAX-traceable pure functions
3. `jax.make_jaxpr(...)`
4. `jax.jit(...).lower(...)`
5. StableHLO IR via `compiler_ir("stablehlo")`

The goal is not SOTA image generation. The goal is to make the compilation pipeline visible and inspectable.

The model uses:

- patch embedding for small images,
- sinusoidal timestep embeddings,
- AdaLN-Zero-style conditioning,
- Transformer blocks,
- a DDPM-style noise-prediction objective.

In [ ]:
# If running in a fresh Colab-like environment, uncomment these lines.
# %pip install -q "jax[cpu]" flax optax

import functools
from dataclasses import dataclass
from typing import Any, Callable, Tuple

import jax
import jax.numpy as jnp
from jax import random

from flax import linen as nn
import optax

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())

## 1. Configuration

We keep the tensor sizes intentionally small so that the resulting `jaxpr` and StableHLO are readable.

The toy input shape is `[batch, height, width, channels] = [2, 16, 16, 1]`.
With a patch size of `4`, the transformer sequence length is:

```text
num_patches = (16 / 4) * (16 / 4) = 16
```

In [ ]:
@dataclass(frozen=True)
class DiTConfig:
    image_size: int = 16
    patch_size: int = 4
    in_channels: int = 1
    hidden_size: int = 64
    depth: int = 2
    num_heads: int = 4
    mlp_ratio: int = 4
    time_embed_dim: int = 128
    diffusion_steps: int = 1_000

cfg = DiTConfig()

assert cfg.image_size % cfg.patch_size == 0
num_patches = (cfg.image_size // cfg.patch_size) ** 2
patch_dim = cfg.patch_size * cfg.patch_size * cfg.in_channels

print("num_patches:", num_patches)
print("patch_dim:", patch_dim)

## 2. Diffusion schedule

A diffusion model learns to predict the noise `eps` added to a clean input `x0`.

Forward noising process:

```text
x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
```

The model receives `(x_t, t)` and predicts `eps`.

In [ ]:
def make_beta_schedule(num_steps: int, beta_start=1e-4, beta_end=2e-2):
    betas = jnp.linspace(beta_start, beta_end, num_steps, dtype=jnp.float32)
    alphas = 1.0 - betas
    alpha_bars = jnp.cumprod(alphas)
    return betas, alphas, alpha_bars

betas, alphas, alpha_bars = make_beta_schedule(cfg.diffusion_steps)


def q_sample(x0: jnp.ndarray, t: jnp.ndarray, noise: jnp.ndarray, alpha_bars: jnp.ndarray) -> jnp.ndarray:
    """Add noise to clean images x0 at integer timesteps t.

    x0:    [B, H, W, C]
    t:     [B]
    noise: [B, H, W, C]
    """
    a_bar = alpha_bars[t]                         # [B]
    a_bar = a_bar[:, None, None, None]            # [B, 1, 1, 1]
    return jnp.sqrt(a_bar) * x0 + jnp.sqrt(1.0 - a_bar) * noise

## 3. Patchify / unpatchify

DiT applies a Transformer to image patches. This is analogous to ViT.

For an image of shape `[B, H, W, C]`, `patchify` returns `[B, N, patch_dim]`, where `N` is the number of patches.

In [ ]:
def patchify(x: jnp.ndarray, patch_size: int) -> jnp.ndarray:
    """Convert images [B, H, W, C] into flattened patches [B, N, P*P*C]."""
    b, h, w, c = x.shape
    p = patch_size
    assert h % p == 0 and w % p == 0
    x = x.reshape(b, h // p, p, w // p, p, c)
    x = jnp.transpose(x, (0, 1, 3, 2, 4, 5))
    return x.reshape(b, (h // p) * (w // p), p * p * c)


def unpatchify(patches: jnp.ndarray, patch_size: int, image_size: int, channels: int) -> jnp.ndarray:
    """Convert flattened patches [B, N, P*P*C] back into images [B, H, W, C]."""
    b, n, d = patches.shape
    p = patch_size
    h = w = image_size
    assert n == (h // p) * (w // p)
    x = patches.reshape(b, h // p, w // p, p, p, channels)
    x = jnp.transpose(x, (0, 1, 3, 2, 4, 5))
    return x.reshape(b, h, w, channels)

# Smoke test
x_test = jnp.arange(2 * 16 * 16 * 1, dtype=jnp.float32).reshape(2, 16, 16, 1)
patches = patchify(x_test, cfg.patch_size)
x_roundtrip = unpatchify(patches, cfg.patch_size, cfg.image_size, cfg.in_channels)
print("patches shape:", patches.shape)
print("roundtrip ok:", jnp.allclose(x_test, x_roundtrip))

## 4. Timestep embedding

The timestep embedding is a sinusoidal embedding followed by an MLP inside the model.

This lets the denoiser know how noisy the current `x_t` is.

In [ ]:
def sinusoidal_timestep_embedding(t: jnp.ndarray, dim: int, max_period: int = 10_000) -> jnp.ndarray:
    """Create sinusoidal embeddings for integer timesteps.

    t: [B]
    returns: [B, dim]
    """
    half = dim // 2
    freqs = jnp.exp(-jnp.log(max_period) * jnp.arange(half, dtype=jnp.float32) / half)
    args = t[:, None].astype(jnp.float32) * freqs[None, :]
    emb = jnp.concatenate([jnp.cos(args), jnp.sin(args)], axis=-1)
    if dim % 2 == 1:
        emb = jnp.pad(emb, ((0, 0), (0, 1)))
    return emb

## 5. DiT components

This is a compact DiT-style block:

- normalize token sequence,
- modulate it with timestep conditioning,
- self-attention,
- MLP,
- residual connections.

The final linear layers in the conditioning path are initialized to zero. This follows the AdaLN-Zero idea: at initialization, the residual modulation begins near zero, which stabilizes diffusion transformer training.

In [ ]:
def modulate(x: jnp.ndarray, shift: jnp.ndarray, scale: jnp.ndarray) -> jnp.ndarray:
    """Apply feature-wise affine modulation.

    x:     [B, N, D]
    shift: [B, D]
    scale: [B, D]
    """
    return x * (1.0 + scale[:, None, :]) + shift[:, None, :]


class MLP(nn.Module):
    hidden_size: int
    out_size: int

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.hidden_size)(x)
        x = nn.gelu(x, approximate=False)
        x = nn.Dense(self.out_size)(x)
        return x


class DiTBlock(nn.Module):
    hidden_size: int
    num_heads: int
    mlp_ratio: int

    @nn.compact
    def __call__(self, x: jnp.ndarray, cond: jnp.ndarray) -> jnp.ndarray:
        # cond produces six vectors for AdaLN-Zero style conditioning:
        # shift/scale/gate for attention and shift/scale/gate for MLP.
        ada = nn.Dense(
            6 * self.hidden_size,
            kernel_init=nn.initializers.zeros,
            bias_init=nn.initializers.zeros,
            name="adaLN_modulation",
        )(nn.silu(cond))
        shift_attn, scale_attn, gate_attn, shift_mlp, scale_mlp, gate_mlp = jnp.split(ada, 6, axis=-1)

        h = nn.LayerNorm(use_bias=False, use_scale=False, name="ln_attn")(x)
        h = modulate(h, shift_attn, scale_attn)
        h = nn.SelfAttention(
            num_heads=self.num_heads,
            qkv_features=self.hidden_size,
            out_features=self.hidden_size,
            name="self_attn",
        )(h)
        x = x + gate_attn[:, None, :] * h

        h = nn.LayerNorm(use_bias=False, use_scale=False, name="ln_mlp")(x)
        h = modulate(h, shift_mlp, scale_mlp)
        h = MLP(self.hidden_size * self.mlp_ratio, self.hidden_size, name="mlp")(h)
        x = x + gate_mlp[:, None, :] * h
        return x


class MiniDiT(nn.Module):
    config: DiTConfig

    @nn.compact
    def __call__(self, x_t: jnp.ndarray, t: jnp.ndarray) -> jnp.ndarray:
        cfg = self.config
        patches = patchify(x_t, cfg.patch_size)  # [B, N, patch_dim]

        x = nn.Dense(cfg.hidden_size, name="patch_embed")(patches)

        # Learned positional embedding. Shape is static and therefore friendly to JIT tracing.
        n = (cfg.image_size // cfg.patch_size) ** 2
        pos = self.param(
            "pos_embed",
            nn.initializers.normal(stddev=0.02),
            (1, n, cfg.hidden_size),
        )
        x = x + pos

        # Timestep conditioning path.
        t_emb = sinusoidal_timestep_embedding(t, cfg.time_embed_dim)
        cond = nn.Dense(cfg.time_embed_dim, name="time_dense_1")(t_emb)
        cond = nn.silu(cond)
        cond = nn.Dense(cfg.hidden_size, name="time_dense_2")(cond)

        for i in range(cfg.depth):
            x = DiTBlock(
                hidden_size=cfg.hidden_size,
                num_heads=cfg.num_heads,
                mlp_ratio=cfg.mlp_ratio,
                name=f"block_{i}",
            )(x, cond)

        # Final AdaLN modulation.
        ada = nn.Dense(
            2 * cfg.hidden_size,
            kernel_init=nn.initializers.zeros,
            bias_init=nn.initializers.zeros,
            name="final_adaLN_modulation",
        )(nn.silu(cond))
        shift, scale = jnp.split(ada, 2, axis=-1)
        x = nn.LayerNorm(use_bias=False, use_scale=False, name="final_ln")(x)
        x = modulate(x, shift, scale)

        # Predict noise in patch space, then unpatchify.
        pred_patches = nn.Dense(cfg.patch_size * cfg.patch_size * cfg.in_channels, name="final_linear")(x)
        pred_noise = unpatchify(pred_patches, cfg.patch_size, cfg.image_size, cfg.in_channels)
        return pred_noise

## 6. Initialize the model

Flax modules are pure once initialized: parameters are explicit inputs to `model.apply(...)`.

This is important for JAX transformations such as `jit`, `grad`, `make_jaxpr`, and lowering to StableHLO.

In [ ]:
rng = random.PRNGKey(0)
rng, init_rng, data_rng, noise_rng = random.split(rng, 4)

model = MiniDiT(cfg)

batch_size = 2
x0 = random.normal(data_rng, (batch_size, cfg.image_size, cfg.image_size, cfg.in_channels))
t = jnp.array([10, 500], dtype=jnp.int32)
noise = random.normal(noise_rng, x0.shape)
x_t = q_sample(x0, t, noise, alpha_bars)

variables = model.init(init_rng, x_t, t)
params = variables["params"]

pred = model.apply({"params": params}, x_t, t)
print("prediction shape:", pred.shape)
print("target noise shape:", noise.shape)

## 7. Loss function and one training step

The loss is the usual DDPM noise-prediction loss:

```text
MSE(model(x_t, t), eps)
```

We keep the training step small and deterministic for inspection.

In [ ]:
def loss_fn(params, x0, t, noise, alpha_bars):
    x_t = q_sample(x0, t, noise, alpha_bars)
    pred_noise = model.apply({"params": params}, x_t, t)
    return jnp.mean((pred_noise - noise) ** 2)

optimizer = optax.adamw(learning_rate=1e-3, weight_decay=1e-4)
opt_state = optimizer.init(params)

@jax.jit
def train_step(params, opt_state, x0, t, noise, alpha_bars):
    loss, grads = jax.value_and_grad(loss_fn)(params, x0, t, noise, alpha_bars)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

params2, opt_state2, loss_value = train_step(params, opt_state, x0, t, noise, alpha_bars)
print("loss:", float(loss_value))

## 8. Inspect `jaxpr`

`jaxpr` is JAX's functional, typed, intermediate representation.

It answers questions such as:

- What primitive operations did JAX trace?
- Which values are constants, inputs, and outputs?
- How did Python control flow become a static computation graph?

The full training-step `jaxpr` is very large, so this notebook first prints a smaller forward-pass `jaxpr`.

In [ ]:
def forward_apply(params, x_t, t):
    return model.apply({"params": params}, x_t, t)

forward_jaxpr = jax.make_jaxpr(forward_apply)(params, x_t, t)
print(forward_jaxpr)

### Smaller jaxpr for patchify + q_sample

For readability, it is often useful to inspect small functions first.

This shows that shape transformations such as `reshape` and `transpose` are also part of the traced program.

In [ ]:
def small_pipeline(x0, t, noise, alpha_bars):
    x_t = q_sample(x0, t, noise, alpha_bars)
    return patchify(x_t, cfg.patch_size)

print(jax.make_jaxpr(small_pipeline)(x0, t, noise, alpha_bars))

## 9. Lower to StableHLO

`jax.jit(fn).lower(...)` performs tracing and lowering without necessarily executing the compiled program.

StableHLO is a portable ML compiler IR. In the JAX pipeline, it is a key representation before backend-specific compilation.

Depending on your JAX version, `compiler_ir("stablehlo")` may return an MLIR module-like object. Printing `str(...)` is usually enough.

In [ ]:
# Forward pass StableHLO.
# This is still fairly long because it includes attention, dense layers, layernorm, and patch operations.
lowered_forward = jax.jit(forward_apply).lower(params, x_t, t)
stablehlo_forward = lowered_forward.compiler_ir(dialect="stablehlo")

print(stablehlo_forward)

## 10. StableHLO for the training step

The training-step StableHLO is much larger because it includes:

- forward pass,
- loss computation,
- reverse-mode autodiff,
- AdamW optimizer update,
- parameter tree update.

This is where JAX's composability becomes visible: `grad`, `jit`, PyTrees, and optimizer logic are all lowered into compiler-visible operations.

In [ ]:
lowered_train_step = train_step.lower(params, opt_state, x0, t, noise, alpha_bars)
stablehlo_train_step = lowered_train_step.compiler_ir(dialect="stablehlo")

# The output can be extremely long. Printing only the first part keeps the notebook usable.
stablehlo_train_step_text = str(stablehlo_train_step)
print(stablehlo_train_step_text[:10_000])
print("\n... truncated ...")
print("StableHLO character length:", len(stablehlo_train_step_text))

## 11. Optional: inspect operation names in StableHLO

This helper extracts rough operation-name counts from the StableHLO text.

It is not a formal parser, but it is useful for orientation.

In [ ]:
import re
from collections import Counter

ops = re.findall(r'"stablehlo\.([a-zA-Z0-9_]+)"', stablehlo_train_step_text)
counts = Counter(ops)
for op, count in counts.most_common(30):
    print(f"{op:30s} {count}")

## 12. Notes on what to look for

When reading the IRs, useful patterns are:

- `reshape` / `transpose`: patchification, unpatchification, attention head layout changes.
- `dot_general`: dense layers and attention projections.
- `gather`: timestep-dependent indexing into `alpha_bars`.
- `broadcast_in_dim`: expanding `[B]` timestep coefficients into image-shaped tensors.
- `sine` / `cosine`: sinusoidal timestep embedding.
- `reduce`: mean, variance, and MSE reductions.
- optimizer operations: AdamW update arithmetic and PyTree-shaped parameter updates.

Conceptually:

```text
Flax Module
  -> pure apply(params, inputs)
  -> JAX tracing
  -> jaxpr
  -> StableHLO
  -> backend-specific executable
```

This is why JAX can compile relatively high-level model code into accelerator-friendly programs.

In [ ]:
# Final quick sanity check: run a few compiled training steps.
params_run = params
opt_state_run = opt_state
rng = random.PRNGKey(42)

for step in range(3):
    rng, x_rng, noise_rng, t_rng = random.split(rng, 4)
    x0_batch = random.normal(x_rng, (batch_size, cfg.image_size, cfg.image_size, cfg.in_channels))
    noise_batch = random.normal(noise_rng, x0_batch.shape)
    t_batch = random.randint(t_rng, (batch_size,), minval=0, maxval=cfg.diffusion_steps, dtype=jnp.int32)
    params_run, opt_state_run, loss_value = train_step(
        params_run, opt_state_run, x0_batch, t_batch, noise_batch, alpha_bars
    )
    print(f"step={step}, loss={float(loss_value):.6f}")